# 05. New Model Test

노트북 셀 안에서 새 모델을 정의하고,
03과 동일한 `run_training` 파이프라인으로 학습/평가한다.
결과는 `artifacts/runs/`·`artifacts/predictions/`에 저장되어
04·06 노트북에서 기존 모델과 함께 비교할 수 있다.

흐름:
1. `DATASET_ID` / `MODEL_ID` / 하이퍼파라미터 스위치 설정
2. 모델 클래스 정의 (셀 안에서 `BaseModel` 상속)
3. `run_training(model_cls=MyModel)` 으로 학습 + 평가 + 예측 저장
4. (선택) Optuna로 간단 탐색

In [1]:
import sys, json, copy
from pathlib import Path
import pandas as pd

_HERE = Path.cwd()
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))
from utils import nb_utils as U
U.add_fuxictr_to_path()

## 1. 스위치

In [2]:
DATASET_ID = 'AmazonElectronics_x1'
MODEL_ID   = 'MyModel_v1'
N_REPEATS  = 1
SEEDS      = [2026, 2027, 2028][:N_REPEATS]

fmap = U.load_feature_map(DATASET_ID)
print(f'dataset={DATASET_ID}  model={MODEL_ID}')
print(f'labels={fmap["labels"]}  total_features={fmap["total_features"]}')

dataset=AmazonElectronics_x1  model=MyModel_v1
labels=['label']  total_features=236971


## 2. 모델 정의

`BaseModel`을 상속하고, `forward(inputs) -> {"y_pred": tensor}` 를 구현하면 된다.
`model_zoo/DIN/src/DIN.py`를 참고해서 작성.

아래 템플릿은 간단한 DNN(embedding → concat → MLP) 예시.

In [3]:
import torch
from torch import nn
from fuxictr.pytorch.models import BaseModel
from fuxictr.pytorch.layers import FeatureEmbeddingDict, MLP_Block


class MyModel(BaseModel):
    def __init__(self,
                 feature_map,
                 model_id='MyModel',
                 gpu=-1,
                 learning_rate=1e-3,
                 embedding_dim=32,
                 hidden_units=[512, 256, 128],
                 hidden_activations='relu',
                 net_dropout=0,
                 batch_norm=False,
                 embedding_regularizer=None,
                 net_regularizer=None,
                 **kwargs):
        super().__init__(feature_map,
                         model_id=model_id,
                         gpu=gpu,
                         embedding_regularizer=embedding_regularizer,
                         net_regularizer=net_regularizer,
                         **kwargs)
        self.embedding_layer = FeatureEmbeddingDict(feature_map, embedding_dim)
        self.dnn = MLP_Block(
            input_dim=feature_map.sum_emb_out_dim(),
            output_dim=1,
            hidden_units=hidden_units,
            hidden_activations=hidden_activations,
            output_activation=self.output_activation,
            dropout_rates=net_dropout,
            batch_norm=batch_norm,
        )
        self.compile(kwargs['optimizer'], kwargs['loss'], learning_rate)
        self.reset_parameters()
        self.model_to_device()

    def forward(self, inputs):
        X = self.get_inputs(inputs)
        feature_emb_dict = self.embedding_layer(X)
        feature_emb = self.embedding_layer.dict2tensor(feature_emb_dict, flatten_emb=True)
        y_pred = self.dnn(feature_emb)
        return {'y_pred': y_pred}


print(f'MyModel defined: {MyModel}')

MyModel defined: <class '__main__.MyModel'>


## 3. 하이퍼파라미터 (dict)

YAML 파일 없이 dict로 직접 지정. `run_training`이 dataset runtime config와 머지.

In [4]:
params = {
    'model': MODEL_ID,
    'task': 'binary_classification',
    'loss': 'binary_crossentropy',
    'metrics': ['AUC', 'logloss'],
    'optimizer': 'adam',
    'learning_rate': 1e-3,
    'embedding_dim': 32,
    'hidden_units': [512, 256, 128],
    'hidden_activations': 'relu',
    'net_dropout': 0,
    'batch_norm': False,
    'embedding_regularizer': 0,
    'net_regularizer': 0,
    'batch_size': 4096,
    'epochs': 10,
    'shuffle': True,
    'monitor': 'AUC',
    'monitor_mode': 'max',
    'early_stop_patience': 2,
    'save_best_only': True,
    'num_workers': 3,
    'verbose': 1,
    'eval_steps': None,
    'debug_mode': False,
    'group_id': None,
    'use_features': None,
    'feature_specs': None,
    'feature_config': None,
    'pickle_feature_encoder': True,
    'gpu': 0,
    'seed': 2026,
}
print(json.dumps({k: v for k, v in params.items() if k not in {'metrics'}}, indent=2, default=str))

{
  "model": "MyModel_v1",
  "task": "binary_classification",
  "loss": "binary_crossentropy",
  "optimizer": "adam",
  "learning_rate": 0.001,
  "embedding_dim": 32,
  "hidden_units": [
    512,
    256,
    128
  ],
  "hidden_activations": "relu",
  "net_dropout": 0,
  "batch_norm": false,
  "embedding_regularizer": 0,
  "net_regularizer": 0,
  "batch_size": 4096,
  "epochs": 10,
  "shuffle": true,
  "monitor": "AUC",
  "monitor_mode": "max",
  "early_stop_patience": 2,
  "save_best_only": true,
  "num_workers": 3,
  "verbose": 1,
  "eval_steps": null,
  "debug_mode": false,
  "group_id": null,
  "use_features": null,
  "feature_specs": null,
  "feature_config": null,
  "pickle_feature_encoder": true,
  "gpu": 0,
  "seed": 2026
}


## 4. 학습 & 평가

In [5]:
summaries = []
for seed in SEEDS:
    p = copy.deepcopy(params)
    p['seed'] = seed
    run_id = U.make_run_id(MODEL_ID, DATASET_ID, seed)
    print(f'\n=== RUN seed={seed}  run_id={run_id} ===')
    summary = U.run_training(
        dataset_id=DATASET_ID,
        model_name=MODEL_ID,
        params=p,
        run_id=run_id,
        save_test_preds=True,
        model_cls=MyModel,
    )
    summaries.append(summary)

pd.DataFrame(summaries)

2026-04-16 12:29:45,995 P2262959 INFO FuxiCTR version: 2.3.9
2026-04-16 12:29:45,997 P2262959 INFO Run 20260416-122945_MyModel_v1_AmazonElectronics_x1_s2026: {
    "batch_norm": "False",
    "batch_size": "4096",
    "data_format": "parquet",
    "data_root": "/NAS/hyunwoong/Hayanmind-project/data",
    "dataset_id": "AmazonElectronics_x1",
    "debug_mode": "False",
    "early_stop_patience": "2",
    "embedding_dim": "32",
    "embedding_regularizer": "0",
    "epochs": "10",
    "eval_steps": "None",
    "feature_config": "None",
    "feature_specs": "None",
    "gpu": "0",
    "group_id": "None",
    "hidden_activations": "relu",
    "hidden_units": "[512, 256, 128]",
    "learning_rate": "0.001",
    "loss": "binary_crossentropy",
    "metrics": "['AUC', 'logloss']",
    "model": "MyModel_v1",
    "model_id": "MyModel_v1_20260416-122945_MyModel_v1_AmazonElectronics_x1_s2026",
    "model_root": "/NAS/hyunwoong/Hayanmind-project/notebook/artifacts/runs",
    "monitor": "AUC",
    "m


=== RUN seed=2026  run_id=20260416-122945_MyModel_v1_AmazonElectronics_x1_s2026 ===


2026-04-16 12:30:24,729 P2262959 INFO Train samples: total/2347550, blocks/1
2026-04-16 12:30:25,786 P2262959 INFO Validation samples: total/261214, blocks/1
2026-04-16 12:30:25,787 P2262959 INFO Loading train and validation data done.
2026-04-16 12:30:26,989 P2262959 INFO Total number of parameters: 7829857.
2026-04-16 12:30:26,991 P2262959 INFO Start training: 574 batches/epoch
2026-04-16 12:30:26,993 P2262959 INFO ************ Epoch=1 start ************


100%|█████████▉| 573/574 [00:15<00:00, 22.16it/s]

2026-04-16 12:30:42,983 P2262959 INFO Train loss: 0.520193
2026-04-16 12:30:42,984 P2262959 INFO Evaluation @epoch 1 - batch 574: 


...

2026-04-16 12:31:41,642 P2262959 INFO [Metrics] AUC: 0.868228 - logloss: 0.455472
2026-04-16 12:31:41,645 P2262959 INFO *** Test evaluation (run=20260416-122945_MyModel_v1_AmazonElectronics_x1_s2026) ***
2026-04-16 12:31:41,646 P2262959 INFO Loading datasets...
2026-04-16 12:31:43,287 P2262959 INFO Test samples: total/384806, blocks/1
2026-04-16 12:31:43,289 P2262959 INFO Loading test data done.


100%|██████████| 94/94 [00:02<00:00, 41.26it/s]


2026-04-16 12:31:45,858 P2262959 INFO [Metrics] AUC: 0.851477 - logloss: 0.483439
2026-04-16 12:31:45,861 P2262959 INFO Loading datasets...
2026-04-16 12:31:47,843 P2262959 INFO Test samples: total/384806, blocks/1
2026-04-16 12:31:47,845 P2262959 INFO Loading test data done.


100%|██████████| 94/94 [00:02<00:00, 36.28it/s]
[preds] saved 384,806 rows -> /NAS/hyunwoong/Hayanmind-project/notebook/artifacts/predictions/AmazonElectronics_x1/MyModel_v1/20260416-122945_MyModel_v1_AmazonElectronics_x1_s2026/preds.parquet
[run 20260416-122945_MyModel_v1_AmazonElectronics_x1_s2026] valid=OrderedDict([('AUC', 0.868228086984713), ('logloss', 0.4554724216158389)])  test=OrderedDict([('AUC', 0.8514770656493564), ('logloss', 0.4834394253556071)])
[run 20260416-122945_MyModel_v1_AmazonElectronics_x1_s2026] artifacts -> /NAS/hyunwoong/Hayanmind-project/notebook/artifacts/runs/AmazonElectronics_x1/MyModel_v1_20260416-122945_MyModel_v1_AmazonElectronics_x1_s2026.*
[run 20260416-122945_MyModel_v1_AmazonElectronics_x1_s2026] preds    -> /NAS/hyunwoong/Hayanmind-project/notebook/artifacts/predictions/AmazonElectronics_x1/MyModel_v1/20260416-122945_MyModel_v1_AmazonElectronics_x1_s2026/preds.parquet


,model,dataset_id,run_id,model_id,train_seconds,valid_AUC,valid_logloss,test_AUC,test_logloss
0,MyModel_v1,AmazonElectronics_x1,20260416-122945_MyModel_v1_AmazonElectronics_x...,MyModel_v1_20260416-122945_MyModel_v1_AmazonEl...,72.57,0.868228,0.455472,0.851477,0.483439


## 5. 전체 run 비교

이전 03 노트북에서 생성한 모델들과 함께 비교.

In [6]:
runs = U.list_runs(DATASET_ID)
cols = [c for c in ['dataset','model','run_id','metric.valid_AUC','metric.test_AUC',
                     'metric.valid_logloss','metric.test_logloss','metric.train_seconds']
        if c in runs.columns]
runs[cols].sort_values('metric.test_AUC' if 'metric.test_AUC' in cols else 'run_id',
                       ascending=False).head(30)

,dataset,model,run_id,metric.valid_AUC,metric.test_AUC,metric.valid_logloss,metric.test_logloss,metric.train_seconds
0,AmazonElectronics_x1,DIN,20260416-121524_DIN_AmazonElectronics_x1_s2026,0.864903,0.852022,0.460568,0.482683,73.11
7,AmazonElectronics_x1,MyModel_v1,20260416-122945_MyModel_v1_AmazonElectronics_x...,0.868228,0.851477,0.455472,0.483439,72.57
4,AmazonElectronics_x1,DIN,tune_t002_s2026,0.863546,0.849563,0.464459,0.512983,55.22
1,AmazonElectronics_x1,DIN,20260416-121659_DIN_AmazonElectronics_x1_s2027,0.862924,0.845535,0.463801,0.520752,97.06
3,AmazonElectronics_x1,DIN,tune_t001_s2026,0.855176,0.839740,0.473655,0.511686,50.13
2,AmazonElectronics_x1,DIN,tune_t000_s2026,0.852111,0.836035,0.478116,0.521070,49.87
6,AmazonElectronics_x1,DIN,tune_t004_s2026,0.854014,0.835699,0.478269,0.551172,73.49
5,AmazonElectronics_x1,DIN,tune_t003_s2026,0.857037,0.834420,0.474442,0.578799,76.36


## 6. (선택) Optuna 간단 탐색

새 모델로 하이퍼파라미터 탐색을 하고 싶다면 아래 셀을 사용.
03과 동일한 패턴이되, `model_cls=MyModel`을 넘긴다.

In [7]:
# --- 필요시 주석 해제 후 사용 ---
#
# import optuna
#
# TUNE_TRIALS = 10
# TUNE_EPOCHS = 3
#
# search_space = {
#     'learning_rate': {'type': 'loguniform', 'low': 1e-4, 'high': 5e-3},
#     'embedding_dim': {'type': 'categorical', 'choices': [16, 32, 64]},
#     'hidden_units': {'type': 'categorical', 'choices': [[256, 128], [512, 256, 128]]},
#     'net_dropout': {'type': 'uniform', 'low': 0.0, 'high': 0.5},
# }
#
# study_dir = U.ARTIFACT_ROOT / 'tuning' / DATASET_ID / MODEL_ID
# study_dir.mkdir(parents=True, exist_ok=True)
#
# def objective(trial):
#     suggested = U.suggest_from_space(trial, search_space)
#     p = copy.deepcopy(params)
#     p.update(suggested)
#     p['epochs'] = TUNE_EPOCHS
#     p['save_best_only'] = False
#     run_id = f'tune_t{trial.number:03d}_s{p["seed"]}'
#     try:
#         summary = U.run_training(
#             dataset_id=DATASET_ID, model_name=MODEL_ID,
#             params=p, run_id=run_id,
#             save_test_preds=False, model_cls=MyModel,
#         )
#     except Exception as e:
#         print(f'[trial {trial.number}] failed: {e!r}')
#         raise optuna.TrialPruned() from e
#     auc = summary.get('valid_AUC')
#     if auc is None:
#         raise optuna.TrialPruned()
#     print(f'[trial {trial.number}] valid_AUC={auc:.5f}')
#     return float(auc)
#
# study = optuna.create_study(
#     study_name=f'{MODEL_ID}__{DATASET_ID}',
#     storage=f'sqlite:///{study_dir / "study.db"}',
#     direction='maximize', load_if_exists=True,
# )
# study.optimize(objective, n_trials=TUNE_TRIALS)
# print('best_value =', study.best_value)
# print('best_params =', study.best_params)